<a href="https://colab.research.google.com/github/jenny4890/deepLearning/blob/main/CNNResidual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import torchvision
import matplotlib.pyplot as plt

if torch.backends.mps.is_available():
    my_device = torch.device('mps')
elif torch.cuda.is_available():
    my_device = torch.device('cuda')
else:
    my_device = torch.device('cpu')

print(my_device)


cuda


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

# Augmentation-> to Tensor -> Normalization.
transform = transforms.Compose([ #여러 변환을 한 번에 적용하는 “파이프라인”
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomCrop(32, padding=4),  #원본이미지에 패딩 4를 채우고 32x32로 크랍한다. CIFAR10 image 32x32다.
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.5),
    transforms.ToTensor(), #이미지(PIL 이미지나 numpy 배열)를PyTorch가 학습할 수 있는 Tensor 형태로 바꿈
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #RGB 평균, 분산 after this data range = -1~1
                                 #음수가 있어 data독립성이 보장되어 역전파 속도가 빨라 렐루도 양수쪽에만 치우치지 않는다.
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)#train=True -> trainset
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)#train=False -> testset
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:14<00:00, 12.0MB/s]


In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self, inchannel, outchannel):
    super().__init__()
# 구조 nn.Conv2d or nn.BatchNorm2d class이고 Params들은 init으로 전달.
#self.conv12, bn12는 Instacnce
#forward에 Instance에 (x) 하는것은 class _call_로 간다.
    self.conv1 = nn.Conv2d(inchannel, outchannel, kernel_size=3, padding = 1, bias = False)
    self.bn1 = nn.BatchNorm2d(outchannel)

    self.conv2 = nn.Conv2d(outchannel, outchannel, kernel_size=3, padding = 1, bias = False)
    self.bn2 = nn.BatchNorm2d(outchannel)

    self.shortcut = nn.Identity()
  # model: [] ->conv/batch/relu [out1] ->conv/batch [out2]
  #         |_________________________________________|^
  def forward(self, x):
    out = F.relu(self.bn1(self.conv1(x))) #out1
    out = self.bn2(self.conv2(out))#out2
    out += self.shortcut(x)
    out = F.relu(out)
    return out

class SuperSimpleResNet(nn.Module):
  def __init__(self, numclasses=10):
    super().__init__()

    self.down_spatial = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True)
    )
    self.features = nn.Sequential(
        ResidualBlock(64,64),
        ResidualBlock(64,64),
        ResidualBlock(64,64),
        ResidualBlock(64,64),
        ResidualBlock(64,64),
    )

    self.gap = nn.AdaptiveAvgPool2d(1)#nn.AdaptiveAvgPool2d(output_size=1)
    #1. 연산 방식각 채널의 $H \times W$ 영역을 격자 모양으로 정확히 4등분(좌상, 우상, 좌하, 우하)한 뒤, 각 등분 구역에 있는 픽셀들의 평균값을 계산합니다.
    #구역마다 1개씩 총 4개의 평균값이 나오게 됩니다.
    #2, 3 등을 쓰는 이유: "특징의 대략적인 위치나 공간적 관계(Where)를 최소한으로 보존해서 넘겨야 한다" (위치 민감형 분류, 구조적 특징 유지 등)
    #. 객체 탐지(Object Detection)나 세그멘테이션(Segmentation)
    self.classifier = nn.Linear(64, numclasses)

  def forward(self, x):
    x = self.down_spatial(x) #output: 배치가 5개면 64x5=320개,(conv연산, )
    x = self.features(x) #output: Residual 이니까. 320개
    x = self.gap(x) # 이것도 채널별로 평균을 1x1로 내는거니까 총: 320개 x 1x1
    x = torch.flatten(x,1) #(B, 64, 1, 1) GAP 통과후 이렇게 되는데, 64 가 차원1 이다. 몇번째 차원부터 시작해서 펼칠것인가. 하는데 여기서는 64부터펼친다.
                          #(5, 64) 가 output
    x = self.classifier(x)# nn.Linear, 입력(5x64) x 가중치(64x 10)
    return x


In [ ]:
net = SuperSimpleResNet(numclasses=10)
lossfn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr = 0.0001)


In [ ]:
net.to(my_device)
num_epochs=100
for epoch in range(num_epochs):
  net.train()
  for batchidx,(data,label) in enumerate(trainloader):
    data, label = data.to(my_device), label.to(my_device)
    output = net(data)
    loss = lossfn(output, label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  net.eval() # dropout(모두 ON) or batchnorm(running avg, variance)을 eval 모드로 동작시켜.
  val_loss=0.0
  correct=0
  with torch.no_grad(): #미분결과 저장하지마 라는 명령 메모리 효육
    for data, label in testloader:
      data, label = data.to(my_device), label.to(my_device)
      output = net(data)
      loss = lossfn(output, label) #error 평균
      val_loss +=loss.item()*data.size(0) # (N, C, H, W)중에 0번째 그러니까 batchsize.
      #loss.item()는 torch.Tensor to 	float변환

      predicted = output.argmax(dim=1) # dim=0 batch 방향, dim=1 가로 방향 class방향.
      #이거 이해 해야함! 만약 배치 사이즈가 3이고 클래스가 4개인 간단한 예시가 있다고 가정해 봅시다. output의 형태는 (3, 4)가 됩니다.
      '''
      (3, 4) 3이 0, 4가 1 dim=0 (세로 방향 / Batch 방향): dim=1 (가로 방향 / Class 방향):
      # [이미지1의 점수 4개], [이미지2의 점수 4개], [이미지3의 점수 4개]
      output = torch.tensor([
      [0.1, 0.2, 0.7, 0.0],  # 이미지 0 (정답 후보들)
      [0.1, 0.9, 0.0, 0.0],  # 이미지 1 (정답 후보들)
      [0.3, 0.3, 0.2, 0.2]   # 이미지 2 (정답 후보들)
      )
      '''
      correct += predicted.eq(label).sum().item() #item()는 torch.Tensor를float변환


  val_loss /= len(testloader.dataset) # 모델 로스
  val_accuracy = 100. * correct / len(testloader.dataset) # 실제 데이타 맞출확률
  print(f"Epoch [{epoch + 1}/{num_epochs}], Training Loss: {loss.item():.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")



Epoch [1/100], Training Loss: 1.0791, Validation Loss: 1.7461, Validation Accuracy: 36.96%
Epoch [2/100], Training Loss: 1.5968, Validation Loss: 1.5915, Validation Accuracy: 41.95%
Epoch [3/100], Training Loss: 1.2309, Validation Loss: 1.5088, Validation Accuracy: 45.78%
Epoch [4/100], Training Loss: 0.6348, Validation Loss: 1.4344, Validation Accuracy: 49.14%
Epoch [5/100], Training Loss: 0.9772, Validation Loss: 1.4029, Validation Accuracy: 50.58%
Epoch [6/100], Training Loss: 1.5080, Validation Loss: 1.3997, Validation Accuracy: 50.55%
Epoch [7/100], Training Loss: 0.5201, Validation Loss: 1.2633, Validation Accuracy: 55.67%
Epoch [8/100], Training Loss: 0.7363, Validation Loss: 1.2756, Validation Accuracy: 54.44%
Epoch [9/100], Training Loss: 0.4404, Validation Loss: 1.2781, Validation Accuracy: 54.70%
Epoch [10/100], Training Loss: 0.8273, Validation Loss: 1.2500, Validation Accuracy: 56.63%
